### Loading the dataset, preprocessing them (resampling) and formating them for the training

In [1]:
import os
import pandas as pd
import numpy as np
import datasets
from transformers import RobertaTokenizer

In [2]:
dataset = "/datasets/values_labels"
labels = [ "Self-direction: thought", "Self-direction: action", "Stimulation",  "Hedonism", "Achievement", "Power: dominance", "Power: resources", "Face", "Security: personal", "Security: societal", "Tradition", "Conformity: rules", "Conformity: interpersonal", "Humility", "Benevolence: caring", "Benevolence: dependability", "Universalism: concern", "Universalism: nature", "Universalism: tolerance", "No Value"]
num_labels = len(labels)

model_name = "roberta-base"
tokenizer = RobertaTokenizer.from_pretrained(model_name)

In [9]:
def create_dataset(directory, output_path, tokenizer, load_labels=True):
    sentences_file_path = os.path.join(directory, "sentences.tsv")
    labels_file_path = os.path.join(directory, "labels.tsv")
    
    data_frame = pd.read_csv(sentences_file_path, encoding="utf-8", sep="\t", header=0)
    encoded_sentences = tokenizer(data_frame["Text"].to_list(), truncation=True)
    
    if load_labels and os.path.isfile(labels_file_path):
        labels_frame = pd.read_csv(labels_file_path, encoding="utf-8", sep="\t", header=0)
        labels_frame = pd.merge(data_frame, labels_frame, on=["Text-ID", "Sentence-ID"], how="inner")
        labels_matrix = np.zeros((labels_frame.shape[0], num_labels))
        for idx, label in enumerate(labels):
            if label in labels_frame.columns:
                labels_matrix[:, idx] = (labels_frame[label] >= 0.5).astype(int)
        encoded_sentences["labels"] = labels_matrix.tolist()
    
    # Save the dataset to disk
    dataset = datasets.Dataset.from_dict(encoded_sentences)
    dataset.save_to_disk(output_path)
    print(f"Saved preprocessed dataset to {output_path}")
    
# Process datasets
directory_test="../datasets/valueeval24/test-english"
directory_train="../datasets/valueeval24/training-english"
directory_validation="../datasets/valueeval24/validation-english"

create_dataset(directory_train, "../datasets/processed_train", tokenizer)
create_dataset(directory_validation, "../datasets/processed_validation", tokenizer)
create_dataset(directory_test, "../datasets/processed_test", tokenizer)

Saving the dataset (0/1 shards):   0%|          | 0/44758 [00:00<?, ? examples/s]

Saved preprocessed dataset to ../datasets/processed_train


Saving the dataset (0/1 shards):   0%|          | 0/14904 [00:00<?, ? examples/s]

Saved preprocessed dataset to ../datasets/processed_validation


Saving the dataset (0/1 shards):   0%|          | 0/14569 [00:00<?, ? examples/s]

Saved preprocessed dataset to ../datasets/processed_test
